In [0]:
%pip install openpyxl

## configuration 

In [0]:
# Databricks notebook: 01_bronze_tax_lifecycle_ingestion

from datetime import datetime, timezone

# Source configuration
SOURCE_PATH = "/Volumes/tx_dev/bronze/source_files/tax_lifecycle_source.xlsx"

# Target configuration
TARGET_TABLE = "tx_dev.bronze.brz_tax_lifecycle"

# Ingestion configuration
SOURCE_SYSTEM = "TAX_REPORTING"
LOAD_TYPE = "FULL"
DRY_RUN = False

print("Tax Lifecycle Bronze Ingestion")
print(f"Source: {SOURCE_PATH}")
print(f"Target: {TARGET_TABLE}")
print(f"Dry run: {DRY_RUN}")

### ## 3. Add a source-file validation cell
This checks that the Excel file exists before we attempt ingestion.

In [0]:
# Validate source file

source_files = [
    file.path.replace("dbfs:", "")
    for file in dbutils.fs.ls(
        "/Volumes/tx_dev/bronze/source_files/"
    )
]

if SOURCE_PATH not in source_files:
    raise FileNotFoundError(
        f"Source file not found: {SOURCE_PATH}"
    )

print("Source file validation successful.")

### Framework-based ingestion (replaces cells 7–20)
Single `ingest_file()` call from `tax_ingestion_framework.py` handles reading, standardization, validation, write, and audit logging.

In [0]:
import importlib
import tax_ingestion_framework
importlib.reload(tax_ingestion_framework)
from tax_ingestion_framework import ingest_file

result = ingest_file(
    spark=spark,
    source_path=SOURCE_PATH,
    target_table=TARGET_TABLE,
    required_columns=[
        "tax_entity_id",
        "tax_entity_name",
        "tax_year",
        "reporting_period",
    ],
    file_format="xlsx",
    sheet_name="Lifecycle",
    source_system=SOURCE_SYSTEM,
    load_type=LOAD_TYPE,
    dry_run=DRY_RUN,
)

print(result)

### 4. Read the Excel file
Excel support can vary by Databricks environment. For this small sample, we'll try Python's pandas and openpyxl to read it, then convert the result to a Spark DataFrame.

In [0]:
# import pandas as pd

# pandas_df = pd.read_excel(
#     SOURCE_PATH,
#     sheet_name="Lifecycle",
#     engine="openpyxl"
# )

# print(f"Rows read: {len(pandas_df)}")
# print(f"Columns: {list(pandas_df.columns)}")

# display(pandas_df)

### 5. Convert to a Spark DataFrame

%pip install openpyxl installs the Excel-reading dependency.
pandas.read_excel() reads the Excel file.
spark.createDataFrame(pandas_df) converts the data into a Spark DataFrame for further processing in Databricks.

In [0]:
# # Convert pandas DataFrame to Spark DataFrame

# df = spark.createDataFrame(pandas_df)

# print(f"Row count: {df.count()}")
# display(df)

### 6: Standardize the data
perform three tasks:
1. Standardize column names.
2. Convert columns to appropriate data types.
3. Check the data before loading it.

Standerised column name
ex. Tax_Entity_ID → tax_entity_id
Tax_Entity_Name → tax_entity_name
Event_Date → event_date

In [0]:
# #1.STANDERISED COLUMN NAMES
# import re

# # Convert column names to lowercase snake_case
# def standardize_column_name(column_name):
#     column_name = column_name.strip()
#     column_name = re.sub(r"[^a-zA-Z0-9]+", "_", column_name)
#     return column_name.strip("_").lower()

# for old_name in df.columns:
#     df = df.withColumnRenamed(
#         old_name,
#         standardize_column_name(old_name)
#     )

# display(df)

6.2 Add ingestion metadata
Every ingested record should carry information about where and when it was loaded.
DataFrame will contain the original business columns plus four metadata columns:
- source_file_name
- source_system
- ingestion_timestamp
- ingestion_date

In [0]:
# from pyspark.sql import functions as F

# df = (
#     df
#     .withColumn("source_file_name", F.lit(SOURCE_PATH.split("/")[-1]))
#     .withColumn(
#         "source_system",
#         F.lit(SOURCE_SYSTEM)
#     )
#     .withColumn(
#         "ingestion_timestamp",
#         F.current_timestamp()
#     )
#     .withColumn(
#         "ingestion_date",
#         F.current_date()
#     )
# )

# display(df)

6.3 Validate the data
Before loading, check that the required fields are present and populated.

In [0]:
# required_columns = [
#     "tax_entity_id",
#     "tax_entity_name",
#     "tax_year",
#     "reporting_period"
# ]

# missing_columns = [
#     column
#     for column in required_columns
#     if column not in df.columns
# ]

# if missing_columns:
#     raise ValueError(
#         f"Required columns are missing: {missing_columns}"
#     )

# null_conditions = [
#     F.col(column).isNull()
#     for column in required_columns
# ]

# invalid_records = df.filter(
#     null_conditions[0]
#     | null_conditions[1]
#     | null_conditions[2]
#     | null_conditions[3]
# )

# invalid_count = invalid_records.count()

# print(f"Total records: {df.count()}")
# print(f"Invalid records: {invalid_count}")

# if invalid_count > 0:
#     display(invalid_records)

7: Write to the Bronze Delta table
initially using overwrite mode because this is our first load. Later, we'll implement configurable load strategies and period-based replacement.

Overwrite replaces the contents of the target table. This is appropriate for our first load, but we'll change the design before using it for recurring production ingestion.

In [0]:
# if not DRY_RUN:
#     (
#         df.write
#         .format("delta")
#         .mode("overwrite")
#         .option("overwriteSchema", "true")
#         .saveAsTable(TARGET_TABLE)
#     )

#     print(f"Successfully loaded data into {TARGET_TABLE}")
# else:
#     print("DRY RUN: Data validated, but no table was written.")

7.V2
Relacing step 7 , adding new code and connenting the old one.
The new code:
- Generates a unique run ID.
- Records the start time.
- Writes the Bronze data.
- Records success or failure.
- Inserts an audit record even if the load fails.

Note: This version logs failed loads when the audit table is writable. If the audit write itself fails, that error can still interrupt the notebook. We'll strengthen the error-handling design later.

In [0]:
# import uuid
# from datetime import datetime, timezone
# from pyspark.sql import functions as F

# # Audit configuration
# AUDIT_TABLE = "tx_dev.control.tax_ingestion_audit"

# run_id = str(uuid.uuid4())
# start_time = datetime.now(timezone.utc)

# source_record_count = df.count()
# target_record_count = 0
# status = "STARTED"
# error_message = None

# try:
#     if not DRY_RUN:
#         (
#             df.write
#             .format("delta")
#             .mode("overwrite")
#             .option("overwriteSchema", "true")
#             .saveAsTable(TARGET_TABLE)
#         )

#         target_record_count = spark.table(TARGET_TABLE).count()

#     status = "SUCCESS"

# except Exception as e:
#     status = "FAILED"
#     error_message = str(e)[:4000]

# finally:
#     end_time = datetime.now(timezone.utc)

#     audit_data = [(
#         run_id,
#         SOURCE_SYSTEM,
#         SOURCE_PATH.split("/")[-1],
#         TARGET_TABLE,
#         LOAD_TYPE,
#         start_time,
#         end_time,
#         source_record_count,
#         target_record_count,
#         status,
#         error_message
#     )]

#     audit_schema = """
#         run_id STRING,
#         source_system STRING,
#         source_file_name STRING,
#         target_table STRING,
#         load_type STRING,
#         start_time TIMESTAMP,
#         end_time TIMESTAMP,
#         source_record_count BIGINT,
#         target_record_count BIGINT,
#         status STRING,
#         error_message STRING
#     """

#     audit_df = spark.createDataFrame(
#         audit_data,
#         schema=audit_schema
#     )

#     (
#         audit_df.write
#         .format("delta")
#         .mode("append")
#         .saveAsTable(AUDIT_TABLE)
#     )

# if status == "FAILED":
#     raise RuntimeError(
#         f"Ingestion failed. Run ID: {run_id}. "
#         f"Error: {error_message}"
#     )

# print(f"Run ID: {run_id}")
# print(f"Status: {status}")
# print(f"Source records: {source_record_count}")
# print(f"Target records: {target_record_count}")

 8: Verify the loaded data

In [0]:
loaded_df = spark.table(TARGET_TABLE)

print(f"Target table: {TARGET_TABLE}")
print(f"Loaded row count: {loaded_df.count()}")

display(loaded_df)

In [0]:
#veryfy to test
display(spark.sql("SELECT * FROM tx_dev.bronze.brz_tax_lifecycle"))

### 9: Create the ingestion audit table
audit logging - project more production-like 
The audit table will track each ingestion run, including its source, target, row counts, status, and any errors. 

9.1 Create the audit table

In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS tx_dev.control.tax_ingestion_audit (
#     run_id STRING,
#     source_system STRING,
#     source_file_name STRING,
#     target_table STRING,
#     load_type STRING,
#     start_time TIMESTAMP,
#     end_time TIMESTAMP,
#     source_record_count BIGINT,
#     target_record_count BIGINT,
#     status STRING,
#     error_message STRING
# )
# USING DELTA;

In [0]:
%sql
----VERIFY AUDIT LOG(CHECK)
SELECT
    run_id,
    source_file_name,
    target_table,
    load_type,
    source_record_count,
    target_record_count,
    status,
    start_time,
    end_time,
    error_message
FROM tx_dev.control.tax_ingestion_audit
ORDER BY start_time DESC;